In [ ]:
import pandas as pd
from kloppy.metrica import load_tracking_epts
import numpy as np

In [ ]:
from pathlib import Path

DATA = Path.cwd().parent / "More Than A Hack 2027 - Initial Challenge - Dataset" / "tactical"

paths_tactical = {
    "adt_xml": DATA / "BAR-ATH_tactical_atd.xml",
    "adt_txt": DATA / "BAR-ATH_tactical_atd.txt",
    "homo":    DATA / "BAR-ATH_tactical_homography.csv",
    "tagging": DATA / "BAR-ATH_tactical_smart_tagging.xml"
}

FPS = 25
halves = {1: (11185, 90651), 2: (110808, 190197)}


In [ ]:
# tracking_data = load_tracking_epts(meta_data=paths_tactical["adt_xml"], raw_data=paths_tactical["adt_txt"])
# df = tracking_data.to_df()
# df.to_csv(DATA / "BAR-ATH_tactical_atd.csv")

In [ ]:
df = pd.read_csv(DATA / "BAR-ATH_tactical_atd.csv")

In [ ]:
df = df.drop("Unnamed: 0", axis=1) if "Unnamed: 0" in list(df.columns) else df # drop unnamed col (idk where it comes from)
df.head(2)

In [ ]:
# split the frames into windows of "ball_present" and "!ball_present"
is_na = df["ball_x"].isna()
change = is_na.ne(is_na.shift())      # true where the value differs from the row before
change.iloc[0] = False                # the first row has no row before it
change_pos = np.flatnonzero(change.to_numpy())   # row positions where a new run starts

windows = {}
prev_index = 0
for num_change, frame in enumerate(change_pos):
    start_index = prev_index
    end_index = frame - 1             # the last frame before the switch
    windows[num_change] = {"start": start_index, "end": end_index, "ball_present": not is_na.iloc[start_index]}
    prev_index = frame

# last window
windows[len(windows)] = {"start": prev_index, "end": len(df) - 1, "ball_present": not is_na.iloc[prev_index]}

runs = pd.DataFrame.from_dict(windows, orient="index")
runs["length"] = runs["end"] - runs["start"] + 1
runs["seconds"] = runs["length"] / FPS
assert runs["length"].sum() == len(df)
assert runs.loc[~runs["ball_present"], "length"].sum() == is_na.sum()
runs

In [ ]:
# cut windows to include only frames within the first or second halfs
rows = []
for half, (h_start, h_end) in halves.items():
    for index, run in runs.iterrows():
        start = max(run["start"], h_start)
        end = min(run["end"], h_end)
        if start <= end:
            length = end - start + 1
            rows.append(
                {
                    "start": start,
                    "end": end,
                    "half": half,
                    "ball_present": run["ball_present"],
                    "length": length,
                    "seconds": length/FPS
                }
            )
windows_cleaned = pd.DataFrame(rows)
gaps = windows_cleaned[windows_cleaned["ball_present"] == False]
gaps

In [ ]:
# test gaps: stretches where the ball is really known
def make_test_gaps(gap_frames, n=300, margin=100, ctx_check=30, seed=0):
    rng = np.random.default_rng(seed)
    miss = np.isnan(orig[:, 0]).astype(int)
    cs = np.r_[0, np.cumsum(miss)]                       # running count of missing frames
    win = gap_frames + 2 * ctx_check                     # gap plus known context both sides

    starts = np.arange(len(orig) - win + 1)
    ok = (cs[starts + win] - cs[starts]) == 0            # windows with no missing frame
    cand = starts[ok] + ctx_check                        # the gap's first frame

    in_half = np.zeros(len(cand), dtype=bool)
    for lo, hi in HALVES:
        in_half |= (cand - ctx_check >= lo) & (cand + gap_frames - 1 + ctx_check <= hi)
    cand = cand[in_half]

    rng.shuffle(cand)
    rows = []
    for s in cand:
        if all(abs(s - a) >= gap_frames + margin for a, _ in rows):   # keep test gaps apart
            rows.append((int(s), int(s) + gap_frames - 1))
            if len(rows) == n:
                break
    print(f"{gap_frames / FPS:.1f} s: {len(cand)} possible starts, using {len(rows)} gaps")
    return rows

# baseline: straight line between the neighbouring known frames
def interp_fill(s, e):
    t = np.arange(s, e + 1)
    return np.column_stack([np.interp(t, [s - 1, e + 1], [orig[s - 1, k], orig[e + 1, k]])
                            for k in range(2)])

# the test: hide, fill, compare with the truth (in metres)
def test(gap_frames, n=300, seed=0):
    rows = make_test_gaps(gap_frames, n=n, seed=seed)
    t = orig.copy()
    for s, e in rows:
        t[s:e + 1] = np.nan
    f, _ = impute(t, pd.DataFrame(rows, columns=["start", "end"]))
    err_model = np.array([np.linalg.norm(f[s:e + 1] - orig[s:e + 1], axis=1).mean()
                          for s, e in rows])
    err_interp = np.array([np.linalg.norm(interp_fill(s, e) - orig[s:e + 1], axis=1).mean()
                           for s, e in rows])
    return err_model, err_interp, rows, f

# table: model vs straight line, per gap length
results = []
for sec in [0.2, 0.5, 1, 2, 3, 5]:
    em, ei, rows, f = test(int(sec * FPS))
    results.append({"gap_s": sec,
                    "model_mean_m": em.mean(), "model_median_m": np.median(em),
                    "interp_mean_m": ei.mean(), "interp_median_m": np.median(ei),
                    "nan_gaps": int(np.isnan(em).sum())})
print(pd.DataFrame(results).round(2).to_string(index=False))

# sweep q (same hidden gaps every time, thanks to the fixed seed)
SHORT_MAX_BACKUP = SHORT_MAX
SHORT_MAX = 0                         # use the Kalman filter for every gap length
for q in [1, 3, 10, 30, 100, 300, 1000]:
    KALMAN_Q = q
    errs = [test(int(sec * FPS))[0].mean() for sec in (0.5, 1, 2, 3, 5)]
    print(f"q={q:5}", np.round(errs, 2))
KALMAN_Q = 20.0
SHORT_MAX = SHORT_MAX_BACKUP          # restore the earlier setting

In [ ]:
# impute missing ball position:
# gap <= 1s -> use linear model
# gap >  1s -> use kalman filter
from sklearn.linear_model import LinearRegression

HALVES = [(11185, 90651), (110808, 190197)]
CTX = 25            # context frames per side
SHORT_MAX = 25      # gaps <= 1 s: linear regression; longer: Kalman
FPS, L, W = 25, 105.3, 68.0
KALMAN_Q, KALMAN_R = 1000, 0.5

def kalman_smooth(z, fps=FPS, q=KALMAN_Q, r=KALMAN_R):
    """Constant-velocity Kalman filter + RTS smoother. z: (T,2) metres, NaN = missing."""
    dt = 1.0 / fps
    T = len(z)

    F = np.eye(4);
    F[0, 2] = F[1, 3] = dt

    H = np.zeros((2, 4))
    H[0, 0] = H[1, 1] = 1

    Q = q * np.array([[dt**3/3, 0, dt**2/2, 0],
                      [0, dt**3/3, 0, dt**2/2],
                      [dt**2/2, 0, dt, 0],
                      [0, dt**2/2, 0, dt]])

    R = r**2 * np.eye(2)

    first = np.flatnonzero(~np.isnan(z[:, 0]))[0]
    x = np.array([z[first, 0], z[first, 1], 0.0, 0.0])
    P = np.diag([50.0**2, 50.0**2, 20.0**2, 20.0**2])   # vague start

    xp = np.zeros((T, 4)); Pp = np.zeros((T, 4, 4))
    xf = np.zeros((T, 4)); Pf = np.zeros((T, 4, 4))

    for t in range(T):
        if t > 0:
            x = F @ x
            P = F @ P @ F.T + Q
        xp[t], Pp[t] = x, P
        if not np.isnan(z[t, 0]):                        # update only if measured
            K = P @ H.T @ np.linalg.inv(H @ P @ H.T + R) # calc gain
            x = x + K @ (z[t] - H @ x) # update estimate
            P = (np.eye(4) - K @ H) @ P # update uncertainty

        xf[t], Pf[t] = x, P

    # smoothing
    xs = xf.copy()
    for t in range(T - 2, -1, -1):
        C = Pf[t] @ F.T @ np.linalg.inv(Pp[t + 1])
        xs[t] = xf[t] + C @ (xs[t + 1] - xp[t + 1])

    return xs                                            # columns: x, y, vx, vy


def fill_linear(orig, s, e, lo, hi, ctx=CTX):
    idx = np.r_[np.arange(max(lo, s - ctx), s), np.arange(e + 1, min(hi, e + ctx) + 1)]
    idx = idx[~np.isnan(orig[idx, 0])]
    if len(idx) == 0:
        return None
    gap = np.arange(s, e + 1)
    if len(idx) == 1:
        return np.tile(orig[idx[0]], (len(gap), 1))
    model = LinearRegression().fit((idx - s).reshape(-1, 1), orig[idx])   # x and y at once
    return model.predict((gap - s).reshape(-1, 1))


def fill_kalman(orig, s, e, lo, hi, ctx=CTX):
    a, b = max(lo, s - ctx), min(hi, e + ctx)
    z = orig[a:b + 1]
    if np.isnan(z[:, 0]).all():
        return None
    return kalman_smooth(z, q=KALMAN_Q, r=KALMAN_R)[s - a:e - a + 1, :2]


def impute(orig, gaps):
    """orig: (N,2) metres with NaN. gaps: DataFrame with start, end (inclusive positions)."""
    filled = orig.copy()
    imputed = np.zeros(len(orig), dtype=bool)
    for _, g in gaps.iterrows():
        s, e = int(g["start"]), int(g["end"])
        lo, hi = next(h for h in halves.values() if h[0] <= s <= h[1])
        pred = (fill_linear if e - s + 1 <= SHORT_MAX else fill_kalman)(orig, s, e, lo, hi)
        if pred is not None:
            filled[s:e + 1] = pred
            imputed[s:e + 1] = True
    return filled, imputed


orig = np.column_stack([df["ball_x"].to_numpy() * L, df["ball_y"].to_numpy() * W])
filled, imputed = impute(orig, gaps)

df["ball_x_imp"] = filled[:, 0] / L
df["ball_y_imp"] = filled[:, 1] / W
df["ball_imputed"] = imputed

In [ ]:
df.to_csv(DATA / "BAR-ATH_tactical_atd_imputed.csv")